## Setup

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the raw data (never edited)
df = pd.read_csv("../data/raw/Solar_Weather_Data_Processed.csv")
df.columns = df.columns.str.strip()
df["timestamp"] = pd.to_datetime(df["Timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

WEATHER = ["GHI", "Temperature", "Relative Humidity", "Wind Speed", "Wind Direction"]
FEATURES = WEATHER + ["Hour", "Month", "SZ Angle", "SA Angle"]   # excludes Day
TARGET = "AC Output"

def evaluate(name, y_true, y_pred):
    """Return MAE, RMSE and R2 for one model."""
    return {
        "Model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }

In [2]:
offset = (df["Hour"] - df["timestamp"].dt.hour) % 24
print("Hour minus Timestamp hour (mod 24), most common values:")
print(offset.value_counts().head(10))

print("\nHour of minimum SZ angle, Hour column:  ", df.groupby("Hour")["SZ Angle"].mean().idxmin())
print("Hour of minimum SZ angle, Timestamp clock:", df.groupby(df["timestamp"].dt.hour)["SZ Angle"].mean().idxmin())

Hour minus Timestamp hour (mod 24), most common values:
13    1520
7      579
12     244
0      175
22     124
17     117
16      13
3        2
5        2
10       1
Name: count, dtype: int64

Hour of minimum SZ angle, Hour column:   12
Hour of minimum SZ angle, Timestamp clock: 23


In [3]:
def shift_scan(d, max_k=12):
    """Correlation between GHI shifted by k hours and AC Output."""
    rows = [(k, d["GHI"].shift(k).corr(d[TARGET])) for k in range(-max_k, max_k + 1)]
    return pd.DataFrame(rows, columns=["shift_h", "corr"])

split = int(len(df) * 0.8)
train_raw, test_raw = df.iloc[:split], df.iloc[split:]

print("Best shifts, TRAIN:")
print(shift_scan(train_raw).sort_values("corr", ascending=False).head(3).round(3).to_string(index=False))
print("\nBest shifts, TEST:")
print(shift_scan(test_raw).sort_values("corr", ascending=False).head(3).round(3).to_string(index=False))

Best shifts, TRAIN:
 shift_h  corr
     -10 0.826
      -9 0.808
     -11 0.752

Best shifts, TEST:
 shift_h  corr
     -10 0.824
      -9 0.778
     -11 0.772


In [4]:
SHIFT = -10   # chosen from the scans above; not tuned on model scores

aligned = df.copy()
aligned["GHI_al"] = df["GHI"].shift(SHIFT)
print("GHI peak hour after shift:", aligned.groupby("Hour")["GHI_al"].mean().idxmax())

GHI peak hour after shift: 12


In [5]:
aligned = aligned.dropna(subset=["GHI_al"]).reset_index(drop=True)
split = int(len(aligned) * 0.8)
tr, te = aligned.iloc[:split], aligned.iloc[split:]
tr_day = tr[tr[TARGET] > 0]
te_day = te[te[TARGET] > 0]

print("Daylight correlation, aligned GHI vs AC Output:",
      round(aligned.loc[aligned[TARGET] > 0, "GHI_al"].corr(aligned.loc[aligned[TARGET] > 0, TARGET]), 3))
print("Daylight correlation, unaligned GHI vs AC Output:",
      round(df.loc[df[TARGET] > 0, "GHI"].corr(df.loc[df[TARGET] > 0, TARGET]), 3))

def hour_corr(g):
    return g["GHI_al"].corr(g[TARGET]) if len(g) > 10 else np.nan

within = tr_day.groupby("Hour").apply(hour_corr)
print("\nWithin-hour correlation (training, daylight):")
print(within.round(2).to_string())
print("Average:", round(within.mean(), 3))

def slope(g):
    g = g[g["GHI_al"] > 100]
    return np.polyfit(g["GHI_al"], g[TARGET], 1)[0]

print("\nSlope of output on GHI, strong-sun rows (GHI > 100):")
print("  Train:", round(slope(tr_day), 3))
print("  Test: ", round(slope(te_day), 3))

Daylight correlation, aligned GHI vs AC Output: 0.706
Daylight correlation, unaligned GHI vs AC Output: -0.467

Within-hour correlation (training, daylight):
Hour
6     0.05
7    -0.00
8     0.06
9     0.19
10   -0.13
11   -0.01
12   -0.07
13    0.03
14   -0.20
15    0.04
16   -0.04
17    0.04
18   -0.14
Average: -0.014

Slope of output on GHI, strong-sun rows (GHI > 100):
  Train: 0.542
  Test:  0.472


In [6]:
def compare_models(shift):
    """Shift the weather columns, split in time order, and score on daylight test rows."""
    d = df.copy()
    d[WEATHER] = df[WEATHER].shift(shift)          # align weather with the power clock
    d = d.dropna(subset=WEATHER).reset_index(drop=True)

    s = int(len(d) * 0.8)                          # 80/20 chronological split
    tr, te = d.iloc[:s], d.iloc[s:]
    print(f"Shift {shift:+d} h | train {len(tr)} rows | test {len(te)} rows")

    mask = (te[TARGET] > 0).values                 # daylight test rows only
    y_true = te[TARGET].values[mask]

    hourly_mean = tr.groupby("Hour")[TARGET].mean()
    rows = [evaluate("Hourly average (baseline)", y_true,
                     te["Hour"].map(hourly_mean).values[mask])]

    for cols, name in [
        (["Hour"], "RF: Hour only"),
        (["Hour", "GHI"], "RF: Hour + GHI"),
        (["Hour", "GHI", "Temperature"], "RF: Hour + GHI + Temperature"),
        (FEATURES, "RF: all features"),
    ]:
        rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
        rf.fit(tr[cols], tr[TARGET])
        rows.append(evaluate(name, y_true, rf.predict(te[cols])[mask]))

    return pd.DataFrame(rows).round(3)

main_table = compare_models(SHIFT)
print(main_table.to_string(index=False))

Shift -10 h | train 2213 rows | test 554 rows
                       Model     MAE    RMSE    R2
   Hourly average (baseline) 107.105 149.446 0.832
               RF: Hour only 107.113 149.526 0.832
              RF: Hour + GHI 130.285 180.310 0.755
RF: Hour + GHI + Temperature 120.173 168.780 0.785
            RF: all features 113.740 151.671 0.827


In [7]:
sensitivity_table = compare_models(-9)
print(sensitivity_table.to_string(index=False))

Shift -9 h | train 2214 rows | test 554 rows
                       Model     MAE    RMSE    R2
   Hourly average (baseline) 107.105 149.446 0.832
               RF: Hour only 107.112 149.525 0.832
              RF: Hour + GHI 125.700 175.866 0.767
RF: Hour + GHI + Temperature 119.748 162.838 0.800
            RF: all features 118.486 156.989 0.814


In [8]:
main_table.to_csv("../models/metrics.csv", index=False)
print("Saved models/metrics.csv with", len(main_table), "rows")

Saved models/metrics.csv with 5 rows
